# Construire un projet de Machine Learning:

## Partie 2 | `Analyse exploratoire des données`

Dans ce notebook, nous allons explorer le jeu de données des ours, en réalisant des statistiques descriptives et des visualisations de données. Nous utiliserons également Streamlit pour ajouter de l’interactivité aux visualisations.

### Ce que nous allons couvrir :

1. **Chargement et préparation des données** - Charger le jeu de données des ours et le préparer pour l’analyse avec Modin (`modin.pandas`) et Snowpark (`snowflake-snowpark-python`)
2. **Statistiques de base** - Calculer et visualiser les statistiques descriptives du jeu de données
3. **Analyse de la distribution des variables** - Explorer la distribution des variables individuelles selon les différentes espèces d’ours avec `Altair` et `Streamlit`
4. **Analyse de corrélation** - Étudier les relations entre les variables numériques à l’aide de cartes de chaleur de corrélation avec `Altair` et `Streamlit`
5. **Relations entre variables** - Visualiser les relations entre paires de variables à l’aide de nuages de points interactifs avec `Altair` et `Streamlit`
6. **Analyse catégorielle** - Examiner la distribution des variables catégorielles, y compris la classification des espèces, avec `Altair` et `Streamlit`



## Installer les bibliothèques prérequises

Les Snowflake Notebooks incluent par défaut des bibliothèques Python courantes. Pour en ajouter d’autres, utilisez le menu déroulant **Packages** en haut à droite.

Ajoutons les packages suivants :
- `modin` - Effectuer des opérations sur les données (lecture/écriture) et du data wrangling comme avec pandas via l’[API Snowpark pandas](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/modin/index)
- `scikit-learn` - Effectuer des découpages de données et construire des modèles de machine learning

# Opérations sur les données

Nous allons commencer par charger et préparer le jeu de données des ours.

## Charger les données

Dans la partie 1, nous avons récupéré, préparé et écrit les données des ours dans Snowflake. Ici, nous allons poursuivre en chargeant les données depuis une table Snowflake.

### Charger les données via une requête SQL

Précédemment, nous les avons enregistrées dans `CHANIN_DEMO_DATA.PUBLIC.BEAR`, nous allons donc interroger les données avec l’instruction SQL suivante :

In [ ]:
SELECT * FROM BEARS_DATA.STAGES.BEAR;

### Charger les données via une instruction Python

Nous pouvons également utiliser la même instruction SQL à l’intérieur de `pd.read_snowflake()`.

In [ ]:
import modin.pandas as pd
import snowflake.snowpark.modin.plugin

df = pd.read_snowflake("SELECT * FROM BEARS_DATA.STAGES.BEAR")
df

Ou simplement en appelant le nom de la table via `pd.read_snowflake()`.

In [ ]:
pd.read_snowflake("BEARS_DATA.STAGES.BEAR")

## Préparation des données

En préparation des graphiques à venir qui nécessitent des données numériques pour les visualisations, nous allons sélectionner uniquement les colonnes numériques du DataFrame en utilisant `select_dtypes()`.

In [ ]:
# Select only numeric columns from the DataFrame
numeric_df = df.select_dtypes(include=['float64', 'int64'])

numeric_df

# Analyse exploratoire des données

L’analyse exploratoire des données (EDA) constitue la première étape essentielle de tout projet de données, au cours de laquelle vous prenez connaissance des données grâce à une exploration ouverte.

Cela se fait en réalisant des statistiques descriptives et des visualisations de données afin de mettre en évidence des motifs, repérer des anomalies et identifier des relations entre les variables, ce qui aide à comprendre les principales caractéristiques du jeu de données et à le préparer pour des tâches plus complexes.

## Distribution des variables
Commençons par explorer la distribution des variables individuelles et voyons comment elles se répartissent selon les différentes espèces d’ours.

Nous allons le faire selon 2 implémentations :
1. Approche classique basée sur un script - Les variables à visualiser peuvent être codées en dur
1. Application de données interactive - Vous pouvez sélectionner la variable à visualiser


In [ ]:
import altair as alt

# Create feature distribution plots
print("--------------------")
print("Feature Distributions")
print("--------------------")

numeric_cols = numeric_df.columns

# Manually specify features by changing the index number
feature = numeric_cols[0]

chart = alt.Chart(df).mark_bar().encode(
    alt.X(f"{feature}:Q", bin=True),
    y='count()',
    color=alt.Color('species:N', scale=alt.Scale(scheme='category10'))
).properties(
    height=380,
    title=f"Distribution of {feature} by Class"
)
chart

In [ ]:
import streamlit as st
import altair as alt

# Create feature distribution plots
st.header("Feature Distributions")

# Create form for feature selection
with st.form("distribution_form"):
    feature = st.selectbox("Select a Feature", numeric_df.columns)
    submit_dist = st.form_submit_button("Submit", type="primary")

if submit_dist:
    chart = alt.Chart(df).mark_bar().encode(
        alt.X(f"{feature}:Q", bin=True),
        y='count()',
        color=alt.Color('Species:N', scale=alt.Scale(scheme='category10'))
    ).properties(
        height=380,
        title=f"Distribution of {feature} by Species"
    )
    st.altair_chart(chart, use_container_width=True)


## Corrélations entre variables

Ensuite, nous allons obtenir une vue d’ensemble de la manière dont toutes les variables numériques sont liées entre elles. Cela nous aide à repérer rapidement les relations les plus intéressantes à explorer plus en détail.



### Matrice de corrélation

Tout d’abord, nous allons calculer une matrice de corrélation. Il s’agit simplement d’un tableau qui montre le coefficient de corrélation (une valeur allant de -1 à +1) entre chaque paire possible de variables numériques. Une valeur de 1 signifie une relation positive parfaite, -1 signifie une relation négative parfaite, et 0 signifie l’absence de relation linéaire.

In [ ]:
# Correlation heatmap
print("--------------------")
print("Feature Correlations")
print("--------------------")

color_option = ['blueorange', 'spectral', 'viridis']
# More color schemes at https://vega.github.io/vega/docs/schemes/

corr_matrix = numeric_df.corr()

corr_data = (
    corr_matrix
    .stack()
    .reset_index(name='value')
    .rename(columns={'level_0': 'index', 'level_1': 'variable'})
)

corr_chart = alt.Chart(corr_data).mark_rect().encode(
    x=alt.X('index:N', sort=None),
    y=alt.Y('variable:N', sort=None),
    color=alt.Color('value:Q', scale=alt.Scale(scheme=color_option[0])),
    tooltip=[alt.Tooltip('index:N'), alt.Tooltip('variable:N'), alt.Tooltip('value:Q')]
).properties(
    width=400,
    title="Correlation Heatmap"
)
corr_chart

In [ ]:
import streamlit as st
import altair as alt

# Create correlation heatmap
st.header("Feature Correlations")

# Add color scheme selection
color_option = st.selectbox("Select Color Scheme", ['blueorange', 'spectral', 'viridis'])

with st.spinner("Generating correlation heatmap..."):
    # Calculate correlation matrix
    corr_matrix = numeric_df.corr()

    # Reshape correlation matrix for visualization 
    corr_data = (
        corr_matrix
        .stack()
        .reset_index(name='value')
        .rename(columns={'level_0': 'index', 'level_1': 'variable'})
    )

    # Create correlation heatmap
    corr_chart = alt.Chart(corr_data).mark_rect().encode(
        x=alt.X('index:N', sort=None),
        y=alt.Y('variable:N', sort=None),
        color=alt.Color('value:Q', scale=alt.Scale(scheme=color_option)),
        tooltip=[alt.Tooltip('index:N'), alt.Tooltip('variable:N'), alt.Tooltip('value:Q')]
    ).properties(
        height=380,
        title="Correlation Heatmap"
    )
    st.altair_chart(corr_chart, use_container_width=True)


## Relations entre variables

Après avoir examiné les distributions des variables individuelles et les corrélations entre variables, explorons plus en détail les relations spécifiques entre des paires de variables. Cette visualisation nous permet de :

- Voir comment différentes variables numériques sont liées entre elles
- Identifier des motifs ou des regroupements potentiels selon les espèces d’ours
- Repérer des valeurs aberrantes ou des relations inhabituelles dans les données

Le nuage de points ci-dessous montre la relation entre deux variables numériques sélectionnées, avec des points colorés selon l’espèce d’ours. Cela nous aide à comprendre comment différentes espèces peuvent se regrouper ou se séparer selon leurs caractéristiques physiques.

In [ ]:
# Scatter plot for feature relationships
print("--------------------")
print("Feature Relationships")
print("--------------------")

# Manually changing the index value
x_axis = numeric_cols[0]
y_axis = numeric_cols[1]

scatter = alt.Chart(df).mark_circle().encode(
    x=f'{x_axis}:Q',
    y=f'{y_axis}:Q',
    color='species:N',
    tooltip=[f'{x_axis}:Q', f'{y_axis}:Q', 'Species:N']
).properties(
    width=380,
    height=380,
    title=f"{x_axis} vs {y_axis} by Class"
)

scatter

In [ ]:
# Generated by Snowflake Copilot
import streamlit as st
import altair as alt

# Create scatter plot for feature relationships
st.header("Feature Relationships")

# Create columns for feature selection
col1, col2 = st.columns(2)
x_axis = col1.selectbox("Select X-axis feature", numeric_df.columns, key="x_feature")
y_axis = col2.selectbox("Select Y-axis feature", numeric_df.columns, key="y_feature") 

with st.spinner("Generating scatter plot..."):
    scatter = alt.Chart(df).mark_circle().encode(
        x=f'{x_axis}:Q',
        y=f'{y_axis}:Q',
        color='species:N',
        tooltip=[f'{x_axis}:Q', f'{y_axis}:Q', 'species:N']
    ).properties(
        height=380,
        title=f"{x_axis} vs {y_axis} by Species"
    )
    st.altair_chart(scatter, use_container_width=True)


## Distribution des classes d’espèces

Examinons la distribution des espèces d’ours dans notre jeu de données. Cette visualisation nous montrera :

- Le nombre d’échantillons pour chaque espèce d’ours
- Si notre jeu de données est équilibré ou déséquilibré entre les différentes espèces
- Les proportions relatives de chaque espèce dans le jeu de données

Ces informations sont essentielles pour comprendre la composition de nos données et les biais potentiels qui pourraient affecter notre analyse.


In [ ]:
# Class distribution
print("--------------------")
print("Species Class Distribution")
print("--------------------")

class_dist = alt.Chart(df).mark_bar().encode(
    x='species:N',
    y='count()',
    color='species:N'
).properties(
    width=400,
    height=400,
    title="Distribution of Bear Classes"
)
class_dist

In [ ]:
import streamlit as st
import altair as alt


# Create categorical distribution plot
st.header("Categorical Feature Distribution")

# Get categorical columns (excluding ID)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != 'id']

# Add visualization options
col1, col2 = st.columns(2)
selected_feature = col1.selectbox("Select Feature", categorical_cols)
chart_type = col2.selectbox("Chart Type", ["Bar", "Pie"])

with st.spinner("Generating distribution plot..."):
    if chart_type == "Bar":
        # Create bar chart
        chart = alt.Chart(df).mark_bar().encode(
            x=f'{selected_feature}:N',
            y='count()',
            color=f'{selected_feature}:N',
            tooltip=[f'{selected_feature}:N', alt.Tooltip('count()', title='Count')]
        ).properties(
            height=380,
            title=f"Distribution of {selected_feature}"
        )
    else:
        # Create pie chart
        chart = alt.Chart(df).mark_arc().encode(
            theta='count()',
            color=f'{selected_feature}:N',
            tooltip=[f'{selected_feature}:N', alt.Tooltip('count()', title='Count')]
        ).properties(
            height=380,
            title=f"Distribution of {selected_feature}"
        )
        
    st.altair_chart(chart, use_container_width=True)


## Script de base

Nous allons maintenant tout regrouper dans une seule cellule de code, que vous pourriez également intégrer dans un seul fichier Python afin de créer un script.

En général, pour afficher du texte en Python, nous pouvons utiliser la méthode `print()`, qui fonctionne bien mais n’est pas très esthétique.

Un notebook Jupyter permet d’utiliser Markdown pour styliser l’affichage du texte.

Comme vous allez bientôt le voir, nous pouvons utiliser des widgets Streamlit à l’intérieur de ce notebook pour y ajouter de l’interactivité.

In [ ]:
print("============================================")
print("Dataset Exploratory Data Analysis")
print("============================================")

# Display basic dataset information
print("--- Dataset Overview ---")
print(f"Total Samples: {len(df)}")
print(f"Number of Features: {len(df.columns) - 1}")
print(f"Number of Species Classes: {len(df['species'].unique())}")

# Display summary statistics
print("--- Summary Statistics ---")
print(f"Average Body Mass: {numeric_df['body_mass_kg'].mean():.2f}")
print(f"Average Shoulder Hump Height: {numeric_df['shoulder_hump_height_cm'].mean():.2f}")
print(f"Average Claw Length: {numeric_df['claw_length_cm'].mean():.2f}")
print(f"Average Snout Length: {numeric_df['snout_length_cm'].mean():.2f}")
print(f"Average Forearm Circumference: {numeric_df['forearm_circumference_cm'].mean():.2f}")
print(f"Average Ear Length: {numeric_df['ear_length_cm'].mean():.2f}")

# Create feature distribution plots
print("--------------------")
print("Feature Distributions")
print("--------------------")

numeric_cols = numeric_df.columns

# Manually specify features by changing the index number
feature = numeric_cols[0]

chart = alt.Chart(df).mark_bar().encode(
    alt.X(f"{feature}:Q", bin=True),
    y='count()',
    color=alt.Color('species:N', scale=alt.Scale(scheme='category10'))
).properties(
    height=380,
    title=f"Distribution of {feature} by Class"
)
chart

# Correlation heatmap
print("--------------------")
print("Feature Correlations")
print("--------------------")

color_option = ['blueorange', 'spectral', 'viridis']
# More color schemes at https://vega.github.io/vega/docs/schemes/

corr_matrix = numeric_df.corr()

corr_data = (
    corr_matrix
    .stack()
    .reset_index(name='value')
    .rename(columns={'level_0': 'index', 'level_1': 'variable'})
)

corr_chart = alt.Chart(corr_data).mark_rect().encode(
    x=alt.X('index:N', sort=None),
    y=alt.Y('variable:N', sort=None),
    color=alt.Color('value:Q', scale=alt.Scale(scheme=color_option[0])),
    tooltip=[alt.Tooltip('index:N'), alt.Tooltip('variable:N'), alt.Tooltip('value:Q')]
).properties(
    width=400,
    title="Correlation Heatmap"
)
corr_chart

# Scatter plot for feature relationships
print("--------------------")
print("Feature Relationships")
print("--------------------")

# Manually changing the index value
x_axis = numeric_cols[0]
y_axis = numeric_cols[1]

scatter = alt.Chart(df).mark_circle().encode(
    x=f'{x_axis}:Q',
    y=f'{y_axis}:Q',
    color='species:N',
    # Corrected line: Add data types to each tooltip field
    tooltip=[f'{x_axis}:Q', f'{y_axis}:Q', 'species:N']
).properties(
    width=380,
    height=380,
    title=f"{x_axis} vs {y_axis} by Class"
)

scatter

# Class distribution
print("--------------------")
print("Class Distribution")
print("--------------------")

class_dist = alt.Chart(df).mark_bar().encode(
    x='species:N',
    y='count()',
    color='species:N'
).properties(
    width=380,
    height=380,
    title="Distribution of Species Classes"
)
class_dist



## Streamlit

Le script précédent, une fois implémenté avec des widgets Streamlit, nous donne une application de données interactive.

In [ ]:
import streamlit as st
import altair as alt
import modin.pandas as pd
import numpy as np

# Set up the page
st.title("🐻 Bear Dataset Exploratory Data Analysis")

# Display basic dataset information
st.header("Dataset Overview")


with st.container(border=True):
    data_col = st.columns(3)
    data_col[0].metric("Total Samples", len(df))
    data_col[1].metric("Number of Features", len(df.columns) - 1)
    data_col[2].metric("Number of Species", len(df['species'].unique()))

# Display summary statistics in a grid
st.header("Summary Statistics")

with st.container(border=True):
    stats_col = st.columns(3)
    stats_col[0].metric("Average Body Mass (kg)", f"{numeric_df['body_mass_kg'].mean():.2f}")
    stats_col[1].metric("Average Shoulder Height (cm)", f"{numeric_df['shoulder_hump_height_cm'].mean():.2f}")
    stats_col[2].metric("Average Claw Length (cm)", f"{numeric_df['claw_length_cm'].mean():.2f}")
    
    stats_col2 = st.columns(3)
    stats_col2[0].metric("Average Snout Length (cm)", f"{numeric_df['snout_length_cm'].mean():.2f}")
    stats_col2[1].metric("Average Forearm Circumference (cm)", f"{numeric_df['forearm_circumference_cm'].mean():.2f}")
    stats_col2[2].metric("Average Ear Length (cm)", f"{numeric_df['ear_length_cm'].mean():.2f}")

# Create feature distribution plots
feature_col_1 = st.columns(2)
feature_col_2 = st.columns(2)

with feature_col_1[0]:
    with st.container(border=True):
        st.header("Feature Distributions")
        numeric_cols = numeric_df.columns
        feature = st.selectbox("Select Feature", numeric_cols)
        
        with st.spinner("Generating feature distribution plot..."):
            chart = alt.Chart(df).mark_bar().encode(
                alt.X(f"{feature}:Q", bin=True),
                y='count()',
                color=alt.Color('species:N', scale=alt.Scale(scheme='category10'))
            ).properties(
                height=380,
                title=f"Distribution of {feature} by Species"
            )
            st.altair_chart(chart, use_container_width=True)

# Correlation heatmap
with feature_col_1[1]:
    with st.container(border=True):
        st.header("Feature Correlations")
        color_option = st.selectbox("Color Scheme:", ['blueorange', 'spectral', 'viridis'])
        
        with st.spinner("Generating correlation heatmap..."):
            corr_matrix = numeric_df.corr()
            corr_data = (
                corr_matrix
                .stack()
                .reset_index(name='value')
                .rename(columns={'level_0': 'index', 'level_1': 'variable'})
            )
    
            corr_chart = alt.Chart(corr_data).mark_rect().encode(
                x=alt.X('index:N', sort=None),
                y=alt.Y('variable:N', sort=None),
                color=alt.Color('value:Q', scale=alt.Scale(scheme=color_option)),
                tooltip=[alt.Tooltip('index:N'), alt.Tooltip('variable:N'), alt.Tooltip('value:Q')]
            ).properties(
                height=380,
                title="Correlation Heatmap"
            )
            st.altair_chart(corr_chart, use_container_width=True)

# Scatter plot for feature relationships
with feature_col_2[0]:
    with st.container(border=True):
        st.header("Feature Relationships")
        axis_col = st.columns(2)
        x_axis = axis_col[0].selectbox("Select X-axis feature", numeric_cols, key="x_axis")
        y_axis = axis_col[1].selectbox("Select Y-axis feature", numeric_cols, key="y_axis")
        
        with st.spinner("Generating scatter plot..."):
            scatter = alt.Chart(df).mark_circle().encode(
                x=f'{x_axis}:Q',
                y=f'{y_axis}:Q',
                color='species:N',
                tooltip=[f'{x_axis}:Q', f'{y_axis}:Q', 'Species:N']
            ).properties(
                height=380,
                title=f"{x_axis} vs {y_axis} by Species"
            )
            st.altair_chart(scatter, use_container_width=True)

# Categorical feature distribution
with feature_col_2[1]:
    with st.container(border=True):
        st.header("Categorical Feature Distribution")
        categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        categorical_cols = [col for col in categorical_cols if col != 'id']
        
        cat_col = st.columns(2)
        selected_feature = cat_col[0].selectbox("Select Feature", categorical_cols)
        chart_type = cat_col[1].selectbox("Chart Type", ["Bar", "Pie"])
        
        with st.spinner("Generating distribution plot..."):
            if chart_type == "Bar":
                chart = alt.Chart(df).mark_bar().encode(
                    x=f'{selected_feature}:N',
                    y='count()',
                    color=f'{selected_feature}:N',
                    tooltip=[f'{selected_feature}:N', alt.Tooltip('count()', title='Count')]
                ).properties(
                    height=380,
                    title=f"Distribution of {selected_feature}"
                )
            else:
                chart = alt.Chart(df).mark_arc().encode(
                    theta='count()',
                    color=f'{selected_feature}:N',
                    tooltip=[f'{selected_feature}:N', alt.Tooltip('count()', title='Count')]
                ).properties(
                    height=380,
                    title=f"Distribution of {selected_feature}"
                )
            st.altair_chart(chart, use_container_width=True)
